# Scheduled context cleanup

Long-running AI agents accumulate conversation history and tool outputs continuously as they operate. Without any management, this accumulation causes two compounding problems: the context window fills with stale, irrelevant information that dilutes the signal of recent and important content, and token costs increase steadily for every request as the agent carries the full weight of its past. A conversation from hours ago about a resolved issue is still consuming tokens and competing for the model's attention against the current task.

Scheduled context cleanup solves this by removing or trimming context at defined intervals - based on how many turns have passed, how old messages are, how large the context has grown, or a combination of these signals. The goal is not to discard everything, but to keep the context window focused and current. Some information, like system instructions or critical user preferences, should always be retained. Other information, like the back-and-forth of a completed subtask, can be safely removed once it is no longer relevant.

This notebook builds up a scheduled cleanup system step by step: starting with the simplest possible approach (keep only the last N messages), then adding time-based expiration, then using an LLM to score message importance for smarter selective retention, and finally combining these signals into a cleanup scheduler that can be dropped into any agent loop.

In [1]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from pydantic import BaseModel, Field
from typing import List, Optional
from datetime import datetime, timedelta
import os

### Initialize the language model

In [2]:
llm = ChatOpenAI(model="gpt-4o-mini-2024-07-18", api_key=os.getenv("OPENAI_API_KEY", "").strip(), temperature=0.0)

## Representing messages with timestamps
Before implementing any cleanup logic, we need a structure that pairs a conversation message with the time it was created. LangChain's built-in message types do not carry timestamps, but we can wrap them in a thin dataclass that adds that information. This wrapper is intentionally minimal - it holds just the message and its creation time, which is everything the cleanup functions will need.

We also build a helper that simulates what a real agent conversation looks like after several hours of operation: a mix of system instructions, user requests, assistant replies, and tool outputs, each created at different points in time.

In [3]:
from dataclasses import dataclass, field

@dataclass
class TimestampedMessage:
    """A conversation message paired with its creation time."""
    message: object          # HumanMessage, AIMessage, SystemMessage, etc.
    created_at: datetime     # When this message was added to the conversation
    pinned: bool = False     # Pinned messages are never removed by cleanup


def simulate_conversation(base_time: datetime) -> List[TimestampedMessage]:
    """Build a sample conversation history spread over several hours."""
    return [
        # System prompt — should survive all cleanup
        TimestampedMessage(
            SystemMessage(content="You are a helpful coding assistant."),
            created_at=base_time - timedelta(hours=8),
            pinned=True,
        ),
        # Old resolved exchange — good candidate for removal
        TimestampedMessage(
            HumanMessage(content="How do I reverse a list in Python?"),
            created_at=base_time - timedelta(hours=7),
        ),
        TimestampedMessage(
            AIMessage(content="Use my_list[::-1] or list.reverse() for in-place reversal."),
            created_at=base_time - timedelta(hours=7),
        ),
        TimestampedMessage(
            HumanMessage(content="Got it, thanks."),
            created_at=base_time - timedelta(hours=6, minutes=55),
        ),
        # Mid-age exchange — borderline depending on threshold
        TimestampedMessage(
            HumanMessage(content="Can you help me write a regex for email validation?"),
            created_at=base_time - timedelta(hours=3),
        ),
        TimestampedMessage(
            AIMessage(content="Sure: r'^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\\.[a-zA-Z]{2,}$'"),
            created_at=base_time - timedelta(hours=3),
        ),
        # Recent active exchange — should always be kept
        TimestampedMessage(
            HumanMessage(content="Now I need to parse a large CSV file efficiently."),
            created_at=base_time - timedelta(minutes=30),
        ),
        TimestampedMessage(
            AIMessage(content="Use pandas read_csv with chunksize for memory-efficient processing."),
            created_at=base_time - timedelta(minutes=28),
        ),
        TimestampedMessage(
            HumanMessage(content="What chunk size should I use?"),
            created_at=base_time - timedelta(minutes=5),
        ),
    ]


now = datetime.now()
conversation = simulate_conversation(now)

print(f"Simulated conversation: {len(conversation)} messages")
print()
for msg in conversation:
    # Calculate how old the message is in minutes for readable display
    age_minutes = (now - msg.created_at).total_seconds() / 60
    role = msg.message.__class__.__name__.replace("Message", "")
    pinned_flag = " [pinned]" if msg.pinned else ""
    print(f"  {role:8s} | {age_minutes:6.0f} min ago{pinned_flag} | {msg.message.content[:60]}")

Simulated conversation: 9 messages

  System   |    480 min ago [pinned] | You are a helpful coding assistant.
  Human    |    420 min ago | How do I reverse a list in Python?
  AI       |    420 min ago | Use my_list[::-1] or list.reverse() for in-place reversal.
  Human    |    415 min ago | Got it, thanks.
  Human    |    180 min ago | Can you help me write a regex for email validation?
  AI       |    180 min ago | Sure: r'^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$'
  Human    |     30 min ago | Now I need to parse a large CSV file efficiently.
  AI       |     28 min ago | Use pandas read_csv with chunksize for memory-efficient proc
  Human    |      5 min ago | What chunk size should I use?


The `TimestampedMessage` dataclass wraps any LangChain message with a `created_at` timestamp and a `pinned` flag. The pinned flag is the simplest form of retention policy: messages marked pinned are excluded from all cleanup operations regardless of age or size. The `simulate_conversation` helper creates a realistic history spread over eight hours - old resolved questions, mid-age exchanges, and a recent ongoing task - so we can observe which messages different cleanup strategies choose to remove.

## Turn-count cleanup: keeping only the last N messages
The simplest possible cleanup strategy is a sliding window: always keep the most recent N messages and drop everything older. This is fast, predictable, and requires no configuration beyond choosing the window size. It works well for chatbots and short-session agents where only the immediate conversational context matters. Pinned messages, like system prompts, are always preserved outside the window count so they are never accidentally evicted.

In [4]:
def cleanup_by_turn_count(
    messages: List[TimestampedMessage],
    max_turns: int,
) -> List[TimestampedMessage]:
    """
    Keep only the most recent `max_turns` non-pinned messages.
    Pinned messages are always retained regardless of window size.
    """
    # Separate pinned messages from the regular sliding window
    pinned = [m for m in messages if m.pinned]
    regular = [m for m in messages if not m.pinned]

    # Keep only the tail of the regular message list
    kept_regular = regular[-max_turns:] if len(regular) > max_turns else regular

    removed_count = len(regular) - len(kept_regular)
    print(f"Turn-count cleanup (max={max_turns}):")
    print(f"  Before: {len(messages)} messages ({len(pinned)} pinned, {len(regular)} regular)")
    print(f"  Removed: {removed_count} oldest regular messages")
    print(f"  After:  {len(pinned) + len(kept_regular)} messages")

    # Pinned messages are prepended so they remain at the front of the history
    return pinned + kept_regular


# Apply a window of 4: keep the 4 most recent non-pinned messages
after_turn_cleanup = cleanup_by_turn_count(conversation, max_turns=4)

print()
print("Remaining messages:")
for msg in after_turn_cleanup:
    role = msg.message.__class__.__name__.replace("Message", "")
    pinned_flag = " [pinned]" if msg.pinned else ""
    print(f"  {role:8s}{pinned_flag} | {msg.message.content[:70]}")

Turn-count cleanup (max=4):
  Before: 9 messages (1 pinned, 8 regular)
  Removed: 4 oldest regular messages
  After:  5 messages

Remaining messages:
  System   [pinned] | You are a helpful coding assistant.
  AI       | Sure: r'^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$'
  Human    | Now I need to parse a large CSV file efficiently.
  AI       | Use pandas read_csv with chunksize for memory-efficient processing.
  Human    | What chunk size should I use?


The function first splits the list into pinned and regular messages. Regular messages are then sliced to their last `max_turns` entries using Python's negative-index slicing - `regular[-max_turns:]` - which always returns the most recent items. Pinned messages are prepended back before returning, ensuring the system prompt stays at position zero as LLMs expect. The removed count is simply the difference between the original and retained regular message counts.

## Time-based cleanup: expiring old messages
Turn-count cleanup is blind to how old messages are - a message from six hours ago and a message from thirty seconds ago count equally. Time-based cleanup instead removes messages that have exceeded a defined age threshold, which better reflects information relevance in long-running agents where yesterday's resolved questions genuinely are no longer useful. Different message types can have different lifetimes: a quick factual question might expire after two hours, while an important design decision might be kept for a full day.

In [5]:
def cleanup_by_age(
    messages: List[TimestampedMessage],
    max_age_hours: float,
    reference_time: Optional[datetime] = None,
) -> List[TimestampedMessage]:
    """
    Remove non-pinned messages older than `max_age_hours`.
    
    Args:
        messages: The current conversation history.
        max_age_hours: Messages older than this are removed.
        reference_time: The time to measure age from (defaults to now).
    """
    if reference_time is None:
        reference_time = datetime.now()

    cutoff = reference_time - timedelta(hours=max_age_hours)
    kept, removed = [], []

    for msg in messages:
        # Pinned messages are never removed regardless of age
        if msg.pinned:
            kept.append(msg)
        elif msg.created_at >= cutoff:
            # Message is within the allowed age window
            kept.append(msg)
        else:
            removed.append(msg)

    print(f"Time-based cleanup (max_age={max_age_hours}h, cutoff={cutoff.strftime('%H:%M')}):")
    print(f"  Removed {len(removed)} messages older than {max_age_hours} hours:")
    for msg in removed:
        age_hours = (reference_time - msg.created_at).total_seconds() / 3600
        role = msg.message.__class__.__name__.replace("Message", "")
        print(f"    - [{role}] {age_hours:.1f}h old: {msg.message.content[:55]}")
    print(f"  Kept {len(kept)} messages")

    return kept


# Expire anything older than 4 hours (the 6-7 hour old messages should go)
after_age_cleanup = cleanup_by_age(conversation, max_age_hours=4)

Time-based cleanup (max_age=4h, cutoff=10:09):
  Removed 3 messages older than 4 hours:
    - [Human] 7.0h old: How do I reverse a list in Python?
    - [AI] 7.0h old: Use my_list[::-1] or list.reverse() for in-place revers
    - [Human] 6.9h old: Got it, thanks.
  Kept 6 messages


The function computes a `cutoff` datetime by subtracting the `max_age_hours` threshold from the reference time. Each message's `created_at` is then compared to this cutoff: messages created before the cutoff are expired and moved to the removed list, while recent messages and all pinned messages are retained. The `reference_time` parameter exists primarily for testing - by passing a fixed time, we can verify which messages would be removed without waiting for real time to pass.

## LLM-based importance scoring for selective retention
Time-based and turn-count cleanup treat all messages uniformly: a message is either inside or outside the threshold. But some messages are more important than their age or position suggests - a user's stated preference, an agreed-upon architectural decision, or a key constraint they mentioned ten turns ago. LLM-based importance scoring gives cleanup a semantic layer: the model evaluates each message in context and assigns it an importance score, allowing cleanup to selectively retain the highest-value messages even when they are old.

The scoring prompt asks the LLM to evaluate a single message against the ongoing task, returning a score from 0 to 10 with a short reason. We then apply a threshold and keep only messages that score above it, along with a fixed number of the most recent turns to ensure immediate context is always preserved.

In [6]:
class ImportanceScore(BaseModel):
    """LLM-assigned importance rating for a single conversation message."""
    score: int = Field(
        description="Importance from 0 (irrelevant) to 10 (critical to keep)"
    )
    reason: str = Field(
        description="One sentence explaining the score"
    )


def score_message_importance(
    message: TimestampedMessage,
    current_task: str,
) -> ImportanceScore:
    """Ask the LLM how important a message is relative to the current task."""
    role = message.message.__class__.__name__.replace("Message", "")
    content = message.message.content

    prompt = f"""You are evaluating whether a past conversation message should be \
kept in an AI agent's context window.

Current task the agent is working on:
{current_task}

Message to evaluate ({role}):
{content}

Rate the importance of retaining this message on a scale from 0 to 10:
- 0-3: Resolved or unrelated, safe to remove
- 4-6: Possibly useful background information
- 7-10: Directly relevant, should be kept

Be concise in your reason."""

    return llm.with_structured_output(ImportanceScore).invoke(prompt)

The `ImportanceScore` Pydantic model tells the LLM exactly what structure to return - an integer score and a one-sentence reason. The prompt frames the decision as a retention question relative to a specific ongoing task, which grounds the scoring in current relevance rather than abstract quality. Using `with_structured_output` ensures the LLM's response is parsed directly into the model without any manual parsing.

In [7]:
def cleanup_by_importance(
    messages: List[TimestampedMessage],
    current_task: str,
    importance_threshold: int = 5,
    always_keep_last_n: int = 3,
) -> List[TimestampedMessage]:
    """
    Retain messages scoring above `importance_threshold` plus the most recent turns.

    Args:
        messages: Conversation history to evaluate.
        current_task: Description of what the agent is currently doing.
        importance_threshold: Minimum score (0-10) to keep a message.
        always_keep_last_n: Most recent non-pinned messages to keep unconditionally.
    """
    pinned = [m for m in messages if m.pinned]
    regular = [m for m in messages if not m.pinned]

    # The last `always_keep_last_n` messages are kept without scoring
    # to guarantee the immediate conversation context is never removed
    recent = regular[-always_keep_last_n:] if len(regular) >= always_keep_last_n else regular
    candidates = regular[:-always_keep_last_n] if len(regular) > always_keep_last_n else []

    print(f"LLM importance cleanup (threshold={importance_threshold}, keep_last={always_keep_last_n}):")
    print(f"  Scoring {len(candidates)} candidate messages...")
    print()

    kept_by_score = []
    for msg in candidates:
        result = score_message_importance(msg, current_task)
        role = msg.message.__class__.__name__.replace("Message", "")
        verdict = "KEEP" if result.score >= importance_threshold else "DROP"
        print(f"  [{verdict}] score={result.score}/10 [{role}]: {msg.message.content[:50]}")
        print(f"          Reason: {result.reason}")

        if result.score >= importance_threshold:
            kept_by_score.append(msg)

    print()
    total_kept = len(pinned) + len(kept_by_score) + len(recent)
    total_removed = len(candidates) - len(kept_by_score)
    print(f"  Removed {total_removed} low-importance messages")
    print(f"  Kept {total_kept} messages ({len(pinned)} pinned, "
          f"{len(kept_by_score)} scored, {len(recent)} recent)")

    # Reassemble: pinned first, then importance-kept older messages, then recent
    return pinned + kept_by_score + recent


current_task = "Help the user parse a large CSV file efficiently using pandas."

after_importance_cleanup = cleanup_by_importance(
    conversation,
    current_task=current_task,
    importance_threshold=5,
    always_keep_last_n=3,
)

LLM importance cleanup (threshold=5, keep_last=3):
  Scoring 5 candidate messages...

  [DROP] score=2/10 [Human]: How do I reverse a list in Python?
          Reason: The message is unrelated to the current task of parsing a CSV file with pandas.
  [DROP] score=4/10 [AI]: Use my_list[::-1] or list.reverse() for in-place r
          Reason: While the message provides a method for reversing lists, it is not directly related to parsing CSV files with pandas.
  [DROP] score=2/10 [Human]: Got it, thanks.
          Reason: The message indicates acknowledgment but adds no relevant information for the task at hand.
  [DROP] score=2/10 [Human]: Can you help me write a regex for email validation
          Reason: The message is unrelated to the current task of parsing a CSV file with pandas.
  [DROP] score=2/10 [AI]: Sure: r'^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z
          Reason: The regex pattern for email validation is not relevant to parsing a CSV file with pandas.

  Removed 5 low-impo

The function splits messages into three buckets before scoring: pinned (never touched), recent (the last `always_keep_last_n` regular messages, kept unconditionally), and candidates (older messages that need to justify their place). Each candidate is scored by the LLM in isolation against the current task description. Messages scoring at or above the threshold are retained; those below are dropped. The final list is reconstructed in chronological order: pinned messages first, then high-scoring older messages, then the always-kept recent tail. The `always_keep_last_n` guard prevents a pathological case where all recent messages happen to receive low scores and the agent loses its immediate context.

## Building a multi-trigger cleanup scheduler
In production, a single cleanup strategy is rarely sufficient on its own. An agent should clean up when any of several conditions are true: after a certain number of turns have passed since the last cleanup, when the total token estimate exceeds a limit, or after a fixed time interval. Checking these conditions manually in every agent loop is tedious and error-prone, so we wrap the logic in a small `CleanupScheduler` that the agent can call after each turn. The scheduler tracks state internally - turn count, last cleanup time - and applies whatever cleanup strategy is appropriate when a trigger fires.

In [8]:
def estimate_tokens(messages: List[TimestampedMessage]) -> int:
    """Rough token estimate: ~4 characters per token across all message content."""
    total_chars = sum(len(msg.message.content) for msg in messages)
    return total_chars // 4


class CleanupScheduler:
    """
    Monitors conversation health and applies cleanup when configured
    thresholds are exceeded. Can be called after every agent turn.
    """

    def __init__(
        self,
        max_turns_between_cleanup: int = 20,
        max_token_estimate: int = 2000,
        max_age_hours: float = 6.0,
        cleanup_strategy: str = "age",  # "age", "turns", or "importance"
        importance_threshold: int = 5,
    ):
        self.max_turns_between_cleanup = max_turns_between_cleanup
        self.max_token_estimate = max_token_estimate
        self.max_age_hours = max_age_hours
        self.cleanup_strategy = cleanup_strategy
        self.importance_threshold = importance_threshold

        # Internal state tracking
        self.turns_since_cleanup = 0
        self.last_cleanup_time = datetime.now()
        self.cleanup_count = 0

    def _should_cleanup(self, messages: List[TimestampedMessage]) -> Optional[str]:
        """Return the name of the first trigger that fires, or None."""
        # Check turn count threshold
        if self.turns_since_cleanup >= self.max_turns_between_cleanup:
            return "turn_count"

        # Check estimated token usage
        if estimate_tokens(messages) >= self.max_token_estimate:
            return "token_limit"

        # Check time elapsed since last cleanup
        hours_elapsed = (datetime.now() - self.last_cleanup_time).total_seconds() / 3600
        if hours_elapsed >= self.max_age_hours:
            return "time_elapsed"

        return None

    def maybe_cleanup(
        self,
        messages: List[TimestampedMessage],
        current_task: str = "",
    ) -> List[TimestampedMessage]:
        """
        Check triggers and apply cleanup if any fires.
        Call this after each agent turn.

        Returns the (possibly cleaned) message list.
        """
        self.turns_since_cleanup += 1

        trigger = self._should_cleanup(messages)
        if trigger is None:
            # No cleanup needed this turn
            return messages

        print(f"  [Scheduler] Trigger fired: '{trigger}' — running {self.cleanup_strategy} cleanup")

        # Apply the configured strategy
        if self.cleanup_strategy == "age":
            messages = cleanup_by_age(messages, max_age_hours=self.max_age_hours)
        elif self.cleanup_strategy == "turns":
            messages = cleanup_by_turn_count(messages, max_turns=self.max_turns_between_cleanup)
        elif self.cleanup_strategy == "importance":
            messages = cleanup_by_importance(
                messages,
                current_task=current_task,
                importance_threshold=self.importance_threshold,
            )

        # Reset counters after a successful cleanup
        self.turns_since_cleanup = 0
        self.last_cleanup_time = datetime.now()
        self.cleanup_count += 1

        return messages

The `CleanupScheduler` separates the decision of *when* to clean up from the decision of *how* to clean up. The `_should_cleanup` method evaluates three independent triggers in priority order and returns the name of the first that fires: a turn counter, a token estimate, and elapsed wall-clock time. The `maybe_cleanup` method is the public interface - the agent calls it after every turn, passing the current message list. If no trigger fires the list is returned unchanged. If a trigger fires, the configured strategy function is called and the counters are reset. The `cleanup_strategy` parameter lets callers choose the removal algorithm without modifying the scheduling logic.

## Simulating the scheduler in an agent loop
To see how the scheduler behaves in practice, we simulate a simple agent loop: the agent receives user messages, appends them to its history, calls the LLM, and then passes the updated history through the scheduler before the next turn. This is the pattern we would apply in any real agent - the scheduler is just one extra call at the end of each iteration. We use a token threshold low enough to fire quickly so the behavior is visible without needing dozens of simulated turns.

In [9]:
# Scheduler: trigger on token count, use age-based cleanup strategy
scheduler = CleanupScheduler(
    max_turns_between_cleanup=10,  # Also clean up after 10 turns
    max_token_estimate=300,        # Fire when estimated tokens exceed 300
    max_age_hours=4.0,             # Age-cleanup removes messages older than 4 hours
    cleanup_strategy="age",
)

# Start from our simulated conversation (already has several old messages)
history = list(conversation)

# New user inputs that drive the continuing conversation
new_user_inputs = [
    "Use 10000 as the chunk size.",
    "How do I combine the chunked results into one DataFrame?",
    "What if the CSV has mixed types in a column?",
]

print("=" * 70)
print(f"Starting agent loop | initial messages: {len(history)} "
      f"(~{estimate_tokens(history)} tokens)")
print("=" * 70)

for i, user_input in enumerate(new_user_inputs, start=1):
    print(f"\n--- Turn {i} ---")
    print(f"User: {user_input}")

    # Append the new user message with the current timestamp
    history.append(TimestampedMessage(
        HumanMessage(content=user_input),
        created_at=datetime.now(),
    ))

    # Call the LLM with the plain message objects (no timestamps)
    lc_messages = [m.message for m in history]
    response = llm.invoke(lc_messages)
    print(f"Assistant: {response.content[:120]}")

    # Append the assistant response to history
    history.append(TimestampedMessage(
        AIMessage(content=response.content),
        created_at=datetime.now(),
    ))

    # Let the scheduler decide if cleanup is needed after this turn
    before_count = len(history)
    before_tokens = estimate_tokens(history)
    history = scheduler.maybe_cleanup(history, current_task=user_input)
    after_count = len(history)

    print(f"  History: {before_count} → {after_count} messages "
          f"(tokens: ~{before_tokens} → ~{estimate_tokens(history)})")

print()
print("=" * 70)
print(f"Final state: {len(history)} messages, ~{estimate_tokens(history)} tokens")
print(f"Total cleanups performed: {scheduler.cleanup_count}")

Starting agent loop | initial messages: 9 (~98 tokens)

--- Turn 1 ---
User: Use 10000 as the chunk size.
Assistant: Using a chunk size of 10,000 is a good starting point for processing large CSV files. You can adjust it based on your sy
  [Scheduler] Trigger fired: 'token_limit' — running age cleanup
Time-based cleanup (max_age=4.0h, cutoff=10:09):
  Removed 3 messages older than 4.0 hours:
    - [Human] 7.0h old: How do I reverse a list in Python?
    - [AI] 7.0h old: Use my_list[::-1] or list.reverse() for in-place revers
    - [Human] 6.9h old: Got it, thanks.
  Kept 8 messages
  History: 11 → 8 messages (tokens: ~308 → ~281)

--- Turn 2 ---
User: How do I combine the chunked results into one DataFrame?
Assistant: You can combine the chunked results into one DataFrame using `pd.concat()`. Here's how you can do it:

1. Read the CSV f
  [Scheduler] Trigger fired: 'token_limit' — running age cleanup
Time-based cleanup (max_age=4.0h, cutoff=10:10):
  Removed 0 messages older than 4.0 h

The agent loop follows a standard pattern: append the user message, call the LLM with the plain message objects (the scheduler's timestamp wrappers are stripped before the API call using a list comprehension), append the response, then call `maybe_cleanup`. The LLM never sees the timestamps - those exist purely for the scheduler's internal logic. The scheduler fires the age-based cleanup when the estimated token count crosses 300, removing the old messages from the simulated history and resetting the counter. The before/after message and token counts printed each turn confirm that cleanup is working and that the context window stays within bounds.

Scheduled context cleanup is a straightforward but essential maintenance operation for any agent that runs beyond a single session. The core design decisions are:
- **What triggers cleanup** - turn count is predictable and easy to reason about; token thresholds directly prevent context overflow; time elapsed is natural for agents that operate over hours or days. Using multiple triggers in combination ensures no single failure mode goes unchecked.
- **What the cleanup strategy removes** - age-based removal is simple and fast with no LLM cost; turn-count windowing is even simpler but ignores content relevance; LLM importance scoring is the most accurate but adds latency and token cost for the scoring calls themselves. Importance scoring is best reserved for cases where context is genuinely valuable and worth the extra inference cost to evaluate carefully.
- **What is never removed** - pinned messages (system prompts, critical instructions) and the most recent N turns should be unconditionally retained. Cleanup that silently removes the system prompt or the current question is worse than no cleanup at all.

The `CleanupScheduler` pattern - a stateful object that tracks turn counts and last-cleanup time, called at the end of every agent iteration - is the recommended integration point. It keeps cleanup logic out of the main agent loop and makes the triggers and strategies independently configurable.